In [ ]:
# ! pip install elasticsearch==8.12.
# ! pip install pandas


In [5]:
# Initialize Elastic Search
from elasticsearch import Elasticsearch
import os
es = Elasticsearch("http://localhost:9200")

In [50]:
import time
import requests

API_URL = "http://localhost:3000/"

# API HELPER FUNCTIONS
def upload_file(filepath):
    url = f"{API_URL}upload"

    with open(filepath, "rb") as f:
        files = {"files": (filepath, f)}

        start = time.perf_counter()
        resp = requests.post(url, files=files)
        end = time.perf_counter()

    latency = (end - start) * 1000
    return resp.json(), latency


def bm25_search(text, k=10):
    start = time.perf_counter()

    response = es.search(
        index="documents",
        body={
            "query": {
                "match": {
                    "content": {
                        "query": text,
                        "boost": 1.0
                    }
                }
            },
            "size": k
        }
    )

    end = time.perf_counter()
    latency = (end - start) * 1000

    results = []
    for hit in response['hits']['hits']:
        results.append({
            "title": hit["_source"]["title"],
            "doc_id": hit["_source"]["doc_id"],
            "_score": hit["_score"]
        })

    return results, latency

def vector_search(text, k=10):
    url = f"{API_URL}search"
    params = {"q": text}
    start = time.perf_counter()
    resp = requests.get(url, params=params)
    end = time.perf_counter()

    latency = (end - start) * 1000

    return resp.json(), latency



In [52]:

# Iterate through samples folder and upload

folder = "Demo_376/samples"

for filename in os.listdir(folder):

    # ensure text document
    if filename.endswith(".txt"):
        path = os.path.join(folder, filename)
        print("Uploading:", filename)

        print(upload_file(path))




Uploading: business_1.txt
({'message': '1 files queued!', 'count': 1}, 134.64309996925294)
Uploading: business_10.txt
({'message': '1 files queued!', 'count': 1}, 17.37479999428615)
Uploading: business_2.txt
({'message': '1 files queued!', 'count': 1}, 22.688499942887574)
Uploading: business_3.txt
({'message': '1 files queued!', 'count': 1}, 11.009500012733042)
Uploading: business_4.txt
({'message': '1 files queued!', 'count': 1}, 12.467100052163005)
Uploading: business_5.txt
({'message': '1 files queued!', 'count': 1}, 9.725800016894937)
Uploading: business_6.txt
({'message': '1 files queued!', 'count': 1}, 10.321999958250672)
Uploading: business_7.txt
({'message': '1 files queued!', 'count': 1}, 11.27929997164756)
Uploading: business_8.txt
({'message': '1 files queued!', 'count': 1}, 10.574600019026548)
Uploading: business_9.txt
({'message': '1 files queued!', 'count': 1}, 11.647299979813397)
Uploading: health_1.txt
({'message': '1 files queued!', 'count': 1}, 9.52230003895238)
Uploa

In [53]:
import pandas as pd

response = es.search(index="documents", query={"match_all": {}}, size=1000)

hits = response["hits"]["hits"]
chunks = [h["_source"] for h in hits]

chunks = pd.DataFrame(chunks)
docs = chunks["doc_id"].unique()

# Corpus Statistics
n_docs = len(docs)
n_chunks = len(chunks)

chunks_per_doc = chunks.groupby("doc_id")


chunk_contents = chunks["content"]

chunks["chunk_word_count"] = chunks["content"].str.split().str.len()
avg_chunk_length = chunk_contents.str.split().str.len().mean()



print(f"Total Documents: {n_docs}")
print(f"Avg Document Size: {round(chunks.groupby("doc_id")["chunk_word_count"].sum().mean(), 2)} words")

print(f"Total Chunks: {n_chunks}")
print(f"Avg Chunk Size: {round(chunks["chunk_word_count"].mean(), 2)} words")

print(f"\nChunks per Document Distribution")
print(f"{chunks_per_doc.size().describe()}")


total_words_per_doc = chunks.groupby("doc_id")["chunk_word_count"].sum()


Total Documents: 50
Avg Document Size: 38.7 words
Total Chunks: 50
Avg Chunk Size: 38.7 words

Chunks per Document Distribution
count    50.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64


In [55]:
import numpy as np

def recall(retrieved, relevant, k):

  # get topk results
  retrieved_k = retrieved[:k]

  # calc
  tp = sum(1 for doc in retrieved_k if doc in relevant)
  return tp / len(relevant) if relevant else 0.0

def precision(retrieved, relevant, k):
  # get topk results
  retrieved_k = retrieved[:k]

  # calc
  tp = sum(1 for doc in retrieved_k if doc in relevant)
  return tp / k

# cause txt files have dumb stuff lol
def strip_template(text):
  lines = text.split('\n')
  cleaned = []

  for line in lines:
      line = line.strip()

      # skip template lines
      if not (line.startswith('Topic:') or
              'discusses various aspects' in line or
              'how X affects our world' in line or
              'Further research into' in line):
          cleaned.append(line)

  return ' '.join(cleaned)

In [56]:

'''
Map documents to topic so we know what results are relevant
'''
topic_docs = {}

for filename in os.listdir(folder):
    # ensure txt document
    if filename.endswith(".txt"):
        with open(os.path.join(folder, filename), "r") as file:
            content = file.read()

        # get topic
        topic = filename.split("_")[0].lower()

        # map docs to topic
        if topic not in topic_docs:
            topic_docs[topic] = []
        topic_docs[topic].append(filename)


'''
IR Evaluation
'''
raw_results = []

# iterate through files
for topic, filenames in topic_docs.items():
    print(f"processing \"{topic}\" files ({len(filenames)})")

    # iterate through files
    for filename in filenames:
        filepath = os.path.join(folder, filename)

        # read document contents as query
        with open(filepath, "r") as file:
            query = file.read().strip()

            query = strip_template(query)

        print(f"querying {filename}")

        # bm25 query
        bm25_results, bm25_latency = bm25_search(query, k=10)
        bm25_docs = [doc["title"] for doc in bm25_results]

        # vector query
        vector_results, vector_latency = vector_search(query, k=10)
        vector_docs = [doc["title"] for doc in vector_results]

        # Store raw data for later processing
        raw_results.append({
            "topic": topic,
            "filename": filename,
            "bm25_docs": bm25_docs,
            "vector_docs": vector_docs,
            "bm25_latency": bm25_latency,
            "vector_latency": vector_latency,
            "filenames": filenames[:]
        })




processing "business" files (10)
querying business_1.txt
querying business_10.txt
querying business_2.txt
querying business_3.txt
querying business_4.txt
querying business_5.txt
querying business_6.txt
querying business_7.txt
querying business_8.txt
querying business_9.txt
processing "health" files (10)
querying health_1.txt
querying health_10.txt
querying health_2.txt
querying health_3.txt
querying health_4.txt
querying health_5.txt
querying health_6.txt
querying health_7.txt
querying health_8.txt
querying health_9.txt
processing "history" files (10)
querying history_1.txt
querying history_10.txt
querying history_2.txt
querying history_3.txt
querying history_4.txt
querying history_5.txt
querying history_6.txt
querying history_7.txt
querying history_8.txt
querying history_9.txt
processing "nature" files (10)
querying nature_1.txt
querying nature_10.txt
querying nature_2.txt
querying nature_3.txt
querying nature_4.txt
querying nature_5.txt
querying nature_6.txt
querying nature_7.txt
que

In [57]:

topic_metrics = {}

for raw in raw_results:
    topic = raw["topic"]
    filenames = raw["filenames"]

    relevant_titles = filenames

    bm25_docs = raw["bm25_docs"]
    vector_docs = raw["vector_docs"]

    # calculate metrics
    bm25_precision = precision(bm25_docs, relevant_titles, 10)
    bm25_recall = recall(bm25_docs, relevant_titles, 10)
    vector_precision = precision(vector_docs, relevant_titles, 10)
    vector_recall = recall(vector_docs, relevant_titles, 10)


    if topic not in topic_metrics:
        topic_metrics[topic] = {
            "bm25_topic_precision": [],
            "bm25_topic_recall": [],
            "vector_topic_precision": [],
            "vector_topic_recall": [],
            "bm25_topic_latency": [],
            "vector_topic_latency": [],
            "files_tested": len(filenames)
        }

    topic_metrics[topic]["bm25_topic_precision"].append(bm25_precision)
    topic_metrics[topic]["bm25_topic_recall"].append(bm25_recall)
    topic_metrics[topic]["vector_topic_precision"].append(vector_precision)
    topic_metrics[topic]["vector_topic_recall"].append(vector_recall)
    topic_metrics[topic]["bm25_topic_latency"].append(raw["bm25_latency"])
    topic_metrics[topic]["vector_topic_latency"].append(raw["vector_latency"])


final_results = []
for topic, metrics in topic_metrics.items():
    final_results.append({
        "topic": topic,
        "bm25_p10": np.mean(metrics["bm25_topic_precision"]),
        "bm25_r10": np.mean(metrics["bm25_topic_recall"]),
        "bm25_latency_ms": np.mean(metrics["bm25_topic_latency"]),
        "vec_p10": np.mean(metrics["vector_topic_precision"]),
        "vec_r10": np.mean(metrics["vector_topic_recall"]),
        "vec_latency_ms": np.mean(metrics["vector_topic_latency"]),
        "files_tested": metrics["files_tested"]
    })

In [59]:
df = pd.DataFrame(final_results)

overall_mean = pd.DataFrame([{
    'topic': 'MEAN',
    'bm25_p10': df['bm25_p10'].mean(),
    'bm25_r10': df['bm25_r10'].mean(),
    'bm25_latency_ms': df['bm25_latency_ms'].mean(),
    'vec_p10': df['vec_p10'].mean(),
    'vec_r10': df['vec_r10'].mean(),
    'vec_latency_ms': df['vec_latency_ms'].mean(),
    'files_tested': df['files_tested'].sum()
}]).round(3)

print(df.round(3))

print("Mean Statistics")
print(overall_mean)


        topic  bm25_p10  bm25_r10  bm25_latency_ms  vec_p10  vec_r10  \
0    business      0.84      0.84           61.531     0.94     0.94   
1      health      0.66      0.66           61.563     0.91     0.91   
2     history      0.93      0.93           62.444     0.87     0.87   
3      nature      0.87      0.87           59.630     0.98     0.98   
4  technology      0.90      0.90           58.054     0.86     0.86   

   vec_latency_ms  files_tested  
0        8778.314            10  
1        7446.992            10  
2        7549.755            10  
3        7406.867            10  
4        7486.392            10  
Mean Statistics
  topic  bm25_p10  bm25_r10  bm25_latency_ms  vec_p10  vec_r10  \
0  MEAN      0.84      0.84           60.644    0.912    0.912   

   vec_latency_ms  files_tested  
0        7733.664            50  


In [ ]:
# # Corpus Statistics
# n_docs = len(docs)
# n_chunks = len(chunks)

# chunks_per_doc = chunks.groupby("doc_id")


# chunk_contents = chunks["content"]

# chunks["chunk_word_count"] = chunks["content"].str.split().str.len()
# avg_chunk_length = chunk_contents.str.split().str.len().mean()



# print(f"Total Documents: {n_docs}")
# print(f"Avg Document Size: {round(chunks.groupby("doc_id")["chunk_word_count"].sum().mean(), 2)} words")

# print(f"Total Chunks: {n_chunks}")
# print(f"Avg Chunk Size: {round(chunks["chunk_word_count"].mean(), 2)} words")

# print(f"\nChunks per Document Distribution")
# print(f"{chunks_per_doc.size().describe()}")


# total_words_per_doc = chunks.groupby("doc_id")["chunk_word_count"].sum()


Total Documents: 53
Avg Document Size: 40.19 words
Total Chunks: 53
Avg Chunk Size: 40.19 words

Chunks per Document Distribution
count    53.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64


In [ ]:
# for filename in filenames[:3]:
#         filepath = os.path.join(folder, filename)

#         # read document contents as query
#         with open(filepath, "r") as file:
#             query = file.read().strip()

#             query = strip_template(query)
#             print(query)


 Artificial Intelligence is revolutionizing how we process data. It is important to understand how technology affects our world today.
 Software development methodologies like Agile improve team efficiency. It is important to understand how technology affects our world today.
 Blockchain technology provides a decentralized way to record transactions. It is important to understand how technology affects our world today.
